# 3.2 Preprocessing

## 3.2.0 Train-test split
We split the training and testing set according to date, we split the data before 2026-05-25 as training set, the data after that as testing set.

In [ ]:
import duckdb
from pathlib import Path

# import library and connect to db
con = duckdb.connect()

In [ ]:
# load dataset
data_path = Path("../data/labeled.parquet")

df = con.sql(f"""
    SELECT *
    FROM read_parquet('{data_path}')
""")

df.limit(5).df()

### Temporal Train-Test Split

In [ ]:
cutoff = "2026-05-25"

train = con.sql(f"""
    SELECT *
    FROM read_parquet('{data_path}')
    WHERE date < DATE '{cutoff}'
""")

test = con.sql(f"""
    SELECT *
    FROM read_parquet('{data_path}')
    WHERE date >= DATE '{cutoff}'
""")

### verify split

In [20]:
print("Training Set:")
print(train.aggregate("COUNT(*) AS n, MIN(date) AS start, MAX(date) AS end").df())

print("\nTesting Set:")
print(test.aggregate("COUNT(*) AS n, MIN(date) AS start, MAX(date) AS end").df())

Training Set:
         n      start        end
0  1882420 2026-01-01 2026-05-24

Testing Set:
        n      start        end
0  479223 2026-05-25 2026-06-30


### Save training and testing set

In [ ]:
con.sql(f"""
    COPY (
        SELECT * FROM read_parquet('{data_path}')
        WHERE date < DATE '{cutoff}'
    )
    TO '../data/train.parquet' (FORMAT PARQUET)
""")

con.sql(f"""
    COPY (
        SELECT * FROM read_parquet('{data_path}')
        WHERE date >= DATE '{cutoff}'
    )
    TO '../data/test.parquet' (FORMAT PARQUET)
""")

## 3.2.1 Feature Selection
For feature selection, we adopted the feature selection method from one of the research papers. We use an adapted changepoint-based feature selection method. The goal is to identify SMART features that show significant changes before disk failure, compared with the changes observed in healthy drives.

we identify the failed drives in the training set and sample healthy drives as a control group. We then extract the daily SMART readings for each selected drive, grouped by drive model and serial number and ordered by date.

In [21]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

DATA = Path("../data")
train_path = DATA / "train.parquet"

# Identify SMART features
smart_cols = [
    c for c in con.sql(
        f"SELECT * FROM '{train_path}' LIMIT 0"
    ).columns
    if c.startswith("smart_")
]

# Identify failed and healthy drives
drive_status = con.sql(f"""
    SELECT model, serial_number,
           MAX(failure) AS failed
    FROM '{train_path}'
    GROUP BY model, serial_number
""").df()

# Use all failed drives and sample healthy controls
groups = []

for model, g in drive_status.groupby("model"):
    failed = g[g["failed"] == 1]
    healthy = g[g["failed"] == 0]

    n_healthy = min(len(healthy), max(40, 2 * len(failed)))

    controls = healthy.sample(
        n=n_healthy, random_state=42
    )

    groups.extend([failed, controls])

drive_ids = pd.concat(groups, ignore_index=True)

con.register(
    "fs_drive_ids",
    drive_ids[["model", "serial_number"]]
)

# Load time series for selected drives
feature_sql = ", ".join(
    f't."{c}"' for c in smart_cols
)

series_df = con.sql(f"""
    SELECT
        t.date,
        t.model,
        t.serial_number,
        t.failure,
        {feature_sql}
    FROM '{train_path}' AS t
    JOIN fs_drive_ids AS i
    USING (model, serial_number)
    ORDER BY t.model, t.serial_number, t.date
""").df()

series_df["date"] = pd.to_datetime(series_df["date"])

print("SMART features:", len(smart_cols))
print("Failed drives:", (drive_ids["failed"] == 1).sum())
print("Healthy controls:", (drive_ids["failed"] == 0).sum())

SMART features: 70
Failed drives: 139
Healthy controls: 278


### Changepoint Detection Function

For each SMART time series, we find the changepoint minimizes the sum of squared errors (SSE) within the two segments. A changepoint is considered valid if it occurs within the specified time window and satisfies the thresholds for SSE reduction.

In [22]:
WINDOW_DAYS = 90
CHANGE_WINDOW_DAYS = 60

MIN_OBS = 20
MIN_SEGMENT = 5

MIN_REDUCTION = 0.30
MIN_MEAN_SHIFT = 0.80
MIN_PERSISTENCE = 0.75


def detect_changepoint(values, dates, anchor_date):
    x = pd.to_numeric(
        values, errors="coerce"
    ).to_numpy(dtype=float)

    # Ignore missing or invalid observations
    valid = np.isfinite(x)
    x = x[valid]
    dates = dates.to_numpy()[valid]

    if len(x) < MIN_OBS:
        return False, False, pd.NaT, np.nan

    # Constant series has no changepoint
    if np.ptp(x) == 0:
        return True, False, pd.NaT, 0.0

    # Standardize within this time series
    # Only for changepoint detection
    z = (x - x.mean()) / x.std()
    n = len(z)

    # Candidate split positions
    k = np.arange(
        MIN_SEGMENT,
        n - MIN_SEGMENT + 1
    )

    sums = np.cumsum(z)
    squares = np.cumsum(z ** 2)

    left_sum = sums[k - 1]
    right_sum = sums[-1] - left_sum

    left_sse = (
        squares[k - 1]
        - left_sum ** 2 / k
    )

    right_sse = (
        squares[-1] - squares[k - 1]
        - right_sum ** 2 / (n - k)
    )

    total_sse = squares[-1] - sums[-1] ** 2 / n
    reductions = total_sse - left_sse - right_sse

    # Best changepoint
    idx = int(k[np.argmax(reductions)])

    improvement = float(
        reductions.max() / total_sse
    )

    cp_date = pd.Timestamp(dates[idx])
    days_before = (anchor_date - cp_date).days

    before = z[:idx]
    after = z[idx:]

    shift = abs(after.mean() - before.mean())

    # Check whether the change persists
    median_before = np.median(before)
    median_after = np.median(after)

    direction = np.sign(
        median_after - median_before
    )

    midpoint = (median_before + median_after) / 2

    persistence = (
        np.mean((after - midpoint) * direction > 0)
        if direction != 0 else 0.0
    )

    changed = (
        1 <= days_before <= CHANGE_WINDOW_DAYS
        and improvement >= MIN_REDUCTION
        and shift >= MIN_MEAN_SHIFT
        and persistence >= MIN_PERSISTENCE
    )

    return (
        True,
        bool(changed),
        cp_date if changed else pd.NaT,
        improvement
    )

### Detect Change Points

We apply the detection function to each SMART feature of every selected drive. For failed drives, we analyze up to 90 days of observations before failure. For healthy drives, we use up to 90 days before their last observation. The results indicate whether a valid changepoint was detected for each combination of drive and feature.

In [23]:
records = []

for (model, serial), g in series_df.groupby(
    ["model", "serial_number"], sort=False
):
    g = g.sort_values("date")

    is_failed = bool(g["failure"].max())

    if is_failed:
        anchor = g.loc[
            g["failure"] == 1, "date"
        ].min()

        # Exclude the actual failure day
        end_mask = g["date"] < anchor
    else:
        anchor = g["date"].max()
        end_mask = g["date"] <= anchor

    # Look at the preceding 90 days
    recent = g[
        end_mask
        & (
            g["date"] >=
            anchor - pd.Timedelta(days=WINDOW_DAYS)
        )
    ]

    for feature in smart_cols:
        eligible, changed, cp_date, improvement = (
            detect_changepoint(
                recent[feature],
                recent["date"],
                anchor
            )
        )

        records.append({
            "model": model,
            "serial_number": serial,
            "failed": is_failed,
            "feature": feature,
            "eligible": eligible,
            "changed": changed,
            "cp_date": cp_date,
            "improvement": improvement
        })

cp_results = pd.DataFrame(records)

print("Evaluated combinations:", len(cp_results))
print("Detected changepoints:", cp_results["changed"].sum())

display(cp_results.head())

Evaluated combinations: 29190
Detected changepoints: 2722


,model,serial_number,failed,feature,eligible,changed,cp_date,improvement
0,HGST HUH721212ALN604,2AGGGPGY,False,smart_1_normalized,True,False,NaT,0.0
1,HGST HUH721212ALN604,2AGGGPGY,False,smart_1_raw,True,False,NaT,0.0
2,HGST HUH721212ALN604,2AGGGPGY,False,smart_2_normalized,True,False,NaT,0.0
3,HGST HUH721212ALN604,2AGGGPGY,False,smart_2_raw,True,False,NaT,0.0
4,HGST HUH721212ALN604,2AGGGPGY,False,smart_3_normalized,True,False,NaT,0.0


### Calculate Changepoint score

For each drive model and SMART feature, we calculate the proportion of failed drives and healthy drives that were detected changepoints. The feature score is calculated as:

##### Score = Failed Changepoint Rate − Healthy Changepoint Rate

A higher score indicates that the feature contains changepoints more frequently in failed drives than in healthy drives.

In [24]:
summary = (
    cp_results
    .groupby(["model", "feature", "failed"])
    .agg(
        eligible=("eligible", "sum"),
        detected=("changed", "sum")
    )
    .reset_index()
)

failed_stats = (
    summary[summary["failed"]]
    [["model", "feature", "eligible", "detected"]]
    .rename(columns={
        "eligible": "failed_eligible",
        "detected": "failed_detected"
    })
)

healthy_stats = (
    summary[~summary["failed"]]
    [["model", "feature", "eligible", "detected"]]
    .rename(columns={
        "eligible": "healthy_eligible",
        "detected": "healthy_detected"
    })
)

ranking = failed_stats.merge(
    healthy_stats,
    on=["model", "feature"]
)

# Require some valid time series in each group
ranking = ranking[
    (ranking["failed_eligible"] >= 5)
    & (ranking["healthy_eligible"] >= 5)
].copy()

ranking["failed_rate"] = (
    ranking["failed_detected"]
    / ranking["failed_eligible"]
)

ranking["healthy_rate"] = (
    ranking["healthy_detected"]
    / ranking["healthy_eligible"]
)

ranking["score"] = (
    ranking["failed_rate"]
    - ranking["healthy_rate"]
)

ranking = ranking.sort_values(
    ["model", "score"],
    ascending=[True, False]
)

display(ranking[
    [
        "model", "feature",
        "failed_detected", "failed_eligible",
        "failed_rate", "healthy_rate", "score"
    ]
].head(20))

,model,feature,failed_detected,failed_eligible,failed_rate,healthy_rate,score
18,HGST HUH721212ALN604,smart_194_normalized,28,45,0.622222,0.102041,0.520181
27,HGST HUH721212ALN604,smart_198_raw,27,45,0.600000,0.081633,0.518367
25,HGST HUH721212ALN604,smart_197_raw,26,45,0.577778,0.091837,0.485941
19,HGST HUH721212ALN604,smart_194_raw,26,45,0.577778,0.132653,0.445125
23,HGST HUH721212ALN604,smart_196_raw,16,45,0.355556,0.000000,0.355556
63,HGST HUH721212ALN604,smart_5_raw,16,45,0.355556,0.000000,0.355556
30,HGST HUH721212ALN604,smart_1_normalized,6,45,0.133333,0.000000,0.133333
31,HGST HUH721212ALN604,smart_1_raw,6,45,0.133333,0.000000,0.133333
14,HGST HUH721212ALN604,smart_192_normalized,5,45,0.111111,0.010204,0.100907
16,HGST HUH721212ALN604,smart_193_normalized,5,45,0.111111,0.010204,0.100907


### Select SMART Features

We select features with a failed changepoint rate of at least 10%, a score of at least 0.05, and changepoints detected in at least two failed drives. We keep up to 10 highest-scoring features for each drive model and take their union to form the final feature set.

In [25]:
TOP_K_PER_MODEL = 10

selected_by_model = (
    ranking[
        (ranking["failed_rate"] >= 0.10)
        & (ranking["score"] >= 0.05)
        & (ranking["failed_detected"] >= 2)
    ]
    .sort_values(
        ["model", "score"],
        ascending=[True, False]
    )
    .groupby("model", group_keys=False)
    .head(TOP_K_PER_MODEL)
)

selected_smart = sorted(
    selected_by_model["feature"].unique().tolist()
)

print("Selected SMART features:", len(selected_smart))
print(selected_smart)

display(selected_by_model[
    ["model", "feature", "failed_rate",
     "healthy_rate", "score"]
])

Selected SMART features: 21
['smart_187_normalized', 'smart_187_raw', 'smart_188_raw', 'smart_190_normalized', 'smart_190_raw', 'smart_192_normalized', 'smart_193_normalized', 'smart_193_raw', 'smart_194_normalized', 'smart_194_raw', 'smart_196_raw', 'smart_197_raw', 'smart_198_raw', 'smart_1_normalized', 'smart_1_raw', 'smart_226_raw', 'smart_3_raw', 'smart_4_raw', 'smart_5_normalized', 'smart_5_raw', 'smart_9_normalized']


,model,feature,failed_rate,healthy_rate,score
18,HGST HUH721212ALN604,smart_194_normalized,0.622222,0.102041,0.520181
27,HGST HUH721212ALN604,smart_198_raw,0.600000,0.081633,0.518367
25,HGST HUH721212ALN604,smart_197_raw,0.577778,0.091837,0.485941
19,HGST HUH721212ALN604,smart_194_raw,0.577778,0.132653,0.445125
23,HGST HUH721212ALN604,smart_196_raw,0.355556,0.000000,0.355556
63,HGST HUH721212ALN604,smart_5_raw,0.355556,0.000000,0.355556
30,HGST HUH721212ALN604,smart_1_normalized,0.133333,0.000000,0.133333
31,HGST HUH721212ALN604,smart_1_raw,0.133333,0.000000,0.133333
14,HGST HUH721212ALN604,smart_192_normalized,0.111111,0.010204,0.100907
16,HGST HUH721212ALN604,smart_193_normalized,0.111111,0.010204,0.100907


### Save feature selection result

In [26]:
def save_selected_dataset(input_path, output_path):
    columns = con.sql(
        f"SELECT * FROM '{input_path}' LIMIT 0"
    ).columns

    # Remove SMART features not selected
    drop_cols = [
        col for col in columns
        if col.startswith("smart_")
        and col not in selected_smart
    ]

    exclude_expr = ", ".join(
        f'"{col}"' for col in drop_cols
    )

    select_expr = (
        f"* EXCLUDE ({exclude_expr})"
        if drop_cols else "*"
    )

    con.execute(f"""
        COPY (
            SELECT {select_expr}
            FROM '{input_path}'
        )
        TO '{output_path}' (FORMAT PARQUET)
    """)


save_selected_dataset(
    "../data/train.parquet",
    "../data/train_selected.parquet"
)

save_selected_dataset(
    "../data/test.parquet",
    "../data/test_selected.parquet"
)

## 3.2.2 Handling Missing Value
In 3.1.3 data quality, we already removed features carry no information (no value for all three models). In this part, we carry out Model-wise median imputation, and provide fixed placeholder values ​​used for SMART attributes that are entirely unavailable for a given model (Structural Missingness).

### Calculate Median 
Use the training set to calculate the median for each SMART feature, grouped by hard drive model.

In [27]:
train_path = "../data/train_selected.parquet"
test_path = "../data/test_selected.parquet"

smart_cols = [
    col for col in con.sql(
        f"SELECT * FROM '{train_path}' LIMIT 0"
    ).columns
    if col.startswith("smart_")
]

print("SMART features to impute:", len(smart_cols))

median_expr = ",\n".join(
    f'median("{col}") AS "{col}"'
    for col in smart_cols
)

con.sql(f"""
    CREATE OR REPLACE TEMP TABLE train_medians AS
    SELECT
        model,
        {median_expr}
    FROM '{train_path}'
    GROUP BY model
""")

SMART features to impute: 21


CountBIGINT
3


### Impute Missing Values
Use the median to impute both training and testing set.

In [28]:
impute_expr = ",\n".join(
    f'COALESCE(t."{col}", m."{col}", 0) AS "{col}"'
    for col in smart_cols
)

def impute_dataset(input_path, output_path):
    con.sql(f"""
        COPY (
            SELECT t.* REPLACE (
                {impute_expr}
            )
            FROM '{input_path}' AS t
            LEFT JOIN train_medians AS m
                ON t.model = m.model
        )
        TO '{output_path}' (FORMAT PARQUET)
    """)

impute_dataset(
    train_path,
    "../data/train_imputed.parquet"
)

impute_dataset(
    test_path,
    "../data/test_imputed.parquet"
)

### Verify Imputation
Verify whether there still any exist missing values.

In [29]:

for dataset in ["train_imputed", "test_imputed"]:
    path = f"../data/{dataset}.parquet"

    missing_expr = " + ".join(
        f'COUNT(*) FILTER (WHERE "{col}" IS NULL)'
        for col in smart_cols
    )

    missing_count = con.execute(f"""
        SELECT {missing_expr}
        FROM '{path}'
    """).fetchone()[0]

    print(f"{dataset}: {missing_count} missing values")


train_imputed: 0 missing values
test_imputed: 0 missing values


## 3.2.3 Normalization and Encoding


For normalizing numerical values, we will normalize the SMART values. We need to consider that we have two different forms of SMART values in our dataset, `smart_x_raw` and `smart_x_normalized`. `smart_x_raw` represents the raw values ​​reported by a hard drive, such as cumulative error counts, total operating hours, temperature, etc. `smart_x_normalized` stands for Values ​​that have been processed by the hard drive manufacturer. We carry out Z-score standardation on `smart_x_raw` and `smart_x_normalized` seperately. Moreover, in 3.1.5 5. Univariate analysis, we found that `smart_5_raw`, `smart_187_raw`, `smart_188_raw`, `smart_197_raw`, and `smart_198_raw` are right-skewed. In this part, we will also use log transformation on these features.

### Normalization

In [30]:
log_cols = {
    "smart_5_raw",
    "smart_187_raw",
    "smart_188_raw",
    "smart_197_raw",
    "smart_198_raw"
}

def feature_expr(alias, col):
    x = f'{alias}."{col}"'
    return f"LN(1.0 + {x})" if col in log_cols else x

# Calculate mean and std using training data only
stats_expr = ",\n".join(
    f'AVG({feature_expr("t", col)}) AS "{col}_mean", '
    f'STDDEV_POP({feature_expr("t", col)}) AS "{col}_std"'
    for col in smart_cols
)

con.execute(f"""
    CREATE OR REPLACE TEMP TABLE smart_scale_stats AS
    SELECT {stats_expr}
    FROM '../data/train_imputed.parquet' AS t
""")


scale_expr = ",\n".join(
    f'''CASE
            WHEN s."{col}_std" > 0
            THEN ({feature_expr("t", col)} - s."{col}_mean")
                 / s."{col}_std"
            ELSE 0.0
        END AS "{col}"'''
    for col in smart_cols
)

def standardize_dataset(input_path, output_path):
    con.execute(f"""
        COPY (
            SELECT t.* REPLACE (
                {scale_expr}
            )
            FROM '{input_path}' AS t
            CROSS JOIN smart_scale_stats AS s
        )
        TO '{output_path}' (FORMAT PARQUET)
    """)

standardize_dataset(
    "../data/train_imputed.parquet",
    "../data/train_scaled.parquet"
)

standardize_dataset(
    "../data/test_imputed.parquet",
    "../data/test_scaled.parquet"
)


### Verify Normalization

In [31]:
cols_to_check = [
    "smart_5_raw",
    "smart_5_normalized",
    "smart_194_raw",
    "smart_194_normalized"
]

check_expr = ",\n".join(
    f'ROUND(AVG("{col}"), 4) AS "{col}_mean", '
    f'ROUND(STDDEV_POP("{col}"), 4) AS "{col}_std"'
    for col in cols_to_check
)

con.sql(f"""
    SELECT {check_expr}
    FROM '../data/train_scaled.parquet'
""").df()


,smart_5_raw_mean,smart_5_raw_std,smart_5_normalized_mean,smart_5_normalized_std,smart_194_raw_mean,smart_194_raw_std,smart_194_normalized_mean,smart_194_normalized_std
0,0.0,1.0,0.0,1.0,0.0,1.0,-0.0,1.0


### One-hot Encoding
We do a one-hot encoding on `model` attribute. It contains three different drive models, `model_ST12000NM0008`, `model_TOSHIBA_MG07ACA14TA`, `model_HGST_HUH721212ALN604`.

In [32]:
import re

models = [
    row[0]
    for row in con.execute("""
        SELECT DISTINCT model
        FROM '../data/train_scaled.parquet'
        WHERE model IS NOT NULL
        ORDER BY model
    """).fetchall()
]

print("Models:", models)

# Generate one-hot encoding expressions
model_columns = {
    model: "model_" + re.sub(r"\W+", "_", model)
    for model in models
}

encoding_expr = ",\n".join(
    f"""CASE WHEN model = '{model.replace("'", "''")}'
        THEN 1 ELSE 0 END AS "{column}" """
    for model, column in model_columns.items()
)

def encode_dataset(input_path, output_path):
    con.execute(f"""
        COPY (
            SELECT
                * EXCLUDE (model),
                {encoding_expr}
            FROM '{input_path}'
        )
        TO '{output_path}' (FORMAT PARQUET)
    """)

encode_dataset(
    "../data/train_scaled.parquet",
    "../data/train_encoded.parquet"
)

encode_dataset(
    "../data/test_scaled.parquet",
    "../data/test_encoded.parquet"
)


Models: ['HGST HUH721212ALN604', 'ST12000NM0008', 'TOSHIBA MG07ACA14TA']


### Verify One-hot encoding

In [33]:
encoded_cols = list(model_columns.values())
col_expr = ", ".join(f'"{col}"' for col in encoded_cols)

for dataset in ["train_encoded", "test_encoded"]:
    print(f"\n{dataset}:")

    result = con.sql(f"""
        SELECT
            {col_expr},
            COUNT(*) AS records
        FROM '../data/{dataset}.parquet'
        GROUP BY {col_expr}
        ORDER BY records DESC
    """).df()

    display(result)



train_encoded:


,model_HGST_HUH721212ALN604,model_ST12000NM0008,model_TOSHIBA_MG07ACA14TA,records
0,0,0,1,1069051
1,0,1,0,531131
2,1,0,0,282238



test_encoded:


,model_HGST_HUH721212ALN604,model_ST12000NM0008,model_TOSHIBA_MG07ACA14TA,records
0,0,0,1,273384
1,0,1,0,135254
2,1,0,0,70585


In [ ]:
# For Testing: Export final preprocessed datasets to CSV


con.execute("""
    COPY (
        SELECT * FROM '../data/train_encoded.parquet'
    )
    TO '../data/train_encoded.csv'
    (FORMAT CSV, HEADER TRUE)
""")

con.execute("""
    COPY (
        SELECT * FROM '../data/test_encoded.parquet'
    )
    TO '../data/test_encoded.csv'
    (FORMAT CSV, HEADER TRUE)
""")

print("Exported train_encoded.csv and test_encoded.csv")

Exported train_encoded.csv and test_encoded.csv
